In [2]:
%matplotlib inline
import sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
from MonteCarlo.MonteCarlo import sample_from_epsilon_ball, monte_carlo_sample_cluster, Cluster

# parent directory to path for imports
sys.path.append(str(Path.cwd().parent))

#  import utility functions from existing modules
from train_cvae import load_data
from ae_utils import _encode, _decode

In [3]:
root = Path.cwd().parent
dataset_type = 'scenairo'

if dataset_type == 'mnist':
    checkpoint_path = root / 'ASAB/runs/0904_mnist/vae_1.pth'
    latent_dim = 12
    data_csv = None

if dataset_type == 'scenairo':
    checkpoint_path = root / 'ASAB/runs/scenairo2004/vae_1.pth'
    latent_dim = 32
    data_csv = None

data = load_data(
    data_csv=data_csv,
    dataset=dataset_type,
    single_class=None,
    data_root=root / 'ASAB/DATA/ScenAIro'
)

X_train_tensor = data['X_train_tensor']
y_train = data['y_train']
input_dim = data['input_dim']
num_classes = data['class_size']
image_shape = data['image_shape']


Class mapping: {'norunway': 0, 'runway': 1}


In [ ]:
##############################
# generate the data in correct structure for the SUT2
# basically runs/scenairo2004/cluster_sampling/decoded_cluster_samples/cluster_0/decoded_images.npy
# the saved numpy data should be (1000 samples, 72, 128, 3) for keras!! this is (1000, 3, 72, 128)
# for the cvae trained with pytorch


target_class = 0

latent_vectors_0 = _encode(target_class,
                        num_classes,
                        input_dim,
                        latent_dim,
                        checkpoint_path,
                        dataset_type,
                        X_train_tensor,
                        y_train,
                        image_shape=image_shape)

# calculate the centroid and the max radius per class in the latent space
## first make numpy
latent_np = latent_vectors_0.detach().cpu().numpy()

center = latent_np.mean(axis=0) # center dim is 32,

cluster_0 = Cluster(latent_np, center)
for epsilon in np.arange(0.1, 1.1, 0.1):

    eps_name = f"{epsilon:.1f}"
    sampled_points = monte_carlo_sample_cluster(cluster_0, epsilon, 1000)
    sampled_points_tensor = torch.tensor(sampled_points, dtype=torch.float32)

    decoded = _decode(
        sampled_points_tensor,
        target_class,
        num_classes,
        input_dim,
        latent_dim,
        checkpoint_path,
        dataset_type,
        image_shape=image_shape,
    )

    # pytorch/CVAE: (1000, 3, 72, 128)
    # Keras/SUT2:  (1000, 72, 128, 3)
    decoded_keras = decoded.permute(0, 2, 3, 1).detach().cpu().numpy().astype(np.float32)


    out_dir = Path(f"runs/scenairo2004/cluster_sampling/epsilon_{eps_name}/decoded_cluster_samples") / f"cluster_{target_class}"
    out_dir.mkdir(parents=True, exist_ok=True)

    np.save(out_dir / "decoded_images.npy", decoded_keras)
    np.save(out_dir / "decode_label.npy", np.array([target_class], dtype=np.int32))


0.1
0.2
0.30000000000000004
0.4
0.5
0.6
0.7000000000000001
0.8
0.9
1.0
(1000, 72, 128, 3)


In [9]:
decoded_keras[0]

array([[[0.35589254, 0.4027886 , 0.48026374],
        [0.34582245, 0.40554723, 0.48036015],
        [0.35068014, 0.3949632 , 0.48374754],
        ...,
        [0.3766309 , 0.410005  , 0.51393306],
        [0.38805795, 0.42711663, 0.5134388 ],
        [0.3639342 , 0.41785842, 0.49879414]],

       [[0.32206336, 0.38470605, 0.47390416],
        [0.32269427, 0.37801912, 0.49155304],
        [0.31931353, 0.37394542, 0.47808862],
        ...,
        [0.35529786, 0.41830817, 0.52010256],
        [0.3711667 , 0.42173433, 0.5297808 ],
        [0.36759284, 0.4024282 , 0.51107675]],

       [[0.30523634, 0.3869215 , 0.5055338 ],
        [0.3165054 , 0.3828136 , 0.4907368 ],
        [0.3146488 , 0.37477413, 0.49410015],
        ...,
        [0.38694894, 0.4407376 , 0.531936  ],
        [0.38451275, 0.4450872 , 0.52363324],
        [0.3700565 , 0.41967323, 0.54170835]],

       ...,

       [[0.31589216, 0.30311337, 0.2805937 ],
        [0.34097007, 0.31141725, 0.2696626 ],
        [0.3147437 , 0